# BookWise — Collaborative Filtering (SVD)
Menggunakan Surprise library + GridSearchCV

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from surprise import SVD, NMF, Dataset, Reader, accuracy
from surprise.model_selection import cross_validate, GridSearchCV, train_test_split
import joblib, os

DATA_DIR  = '../../data'
MODEL_DIR = '../model'

ratings = pd.read_csv(f'{DATA_DIR}/BX-Book-Ratings.csv', sep=';', encoding='latin-1', on_bad_lines='skip')
ratings.columns = ratings.columns.str.strip().str.replace('"', '')

# Filter: hanya rating eksplisit, user & buku dengan cukup data
ratings = ratings[ratings['Book-Rating'] > 0]
user_counts = ratings['User-ID'].value_counts()
book_counts = ratings['ISBN'].value_counts()
ratings = ratings[ratings['User-ID'].isin(user_counts[user_counts >= 5].index)]
ratings = ratings[ratings['ISBN'].isin(book_counts[book_counts >= 5].index)]
print(f'Filtered ratings: {len(ratings):,}')

In [ ]:
reader = Reader(rating_scale=(1, 10))
data   = Dataset.load_from_df(ratings[['User-ID', 'ISBN', 'Book-Rating']], reader)

# Hyperparameter tuning
param_grid = {
    'n_epochs': [20, 30],
    'lr_all':   [0.005, 0.01],
    'reg_all':  [0.02, 0.1],
}
print('Running GridSearchCV...')
gs = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=3, n_jobs=-1)
gs.fit(data)

print(f'Best RMSE: {gs.best_score["rmse"]:.4f}')
print(f'Best MAE:  {gs.best_score["mae"]:.4f}')
print(f'Best params: {gs.best_params["rmse"]}')

In [ ]:
# Cross-validation dengan best params
best_params = gs.best_params['rmse']
cv_results  = cross_validate(SVD(**best_params), data, measures=['rmse', 'mae'], cv=5, verbose=True)

print(f'\nMean RMSE: {cv_results["test_rmse"].mean():.4f} ± {cv_results["test_rmse"].std():.4f}')
print(f'Mean MAE:  {cv_results["test_mae"].mean():.4f} ± {cv_results["test_mae"].std():.4f}')

# Plot RMSE per fold
plt.figure(figsize=(8, 4))
plt.plot(range(1, 6), cv_results['test_rmse'], marker='o', label='RMSE')
plt.plot(range(1, 6), cv_results['test_mae'],  marker='s', label='MAE')
plt.xlabel('Fold')
plt.ylabel('Error')
plt.title('Cross-Validation Results (SVD)')
plt.legend()
plt.tight_layout()
plt.savefig('cv_results.png', dpi=150)
plt.show()

In [ ]:
# Bandingkan SVD vs NMF
print('Comparing SVD vs NMF...')
for algo_class, name in [(SVD, 'SVD'), (NMF, 'NMF')]:
    res = cross_validate(algo_class(), data, measures=['rmse'], cv=3)
    print(f'{name} RMSE: {res["test_rmse"].mean():.4f}')

# Train final model
trainset = data.build_full_trainset()
cf_model = SVD(**best_params)
cf_model.fit(trainset)
joblib.dump(cf_model, f'{MODEL_DIR}/collaborative.pkl')
print('✅ Collaborative model saved.')